# Phase 3 — Train the Skin-Condition Classifier on SCIN

Open in Colab (Runtime → Change runtime type → **T4 GPU**).

End-to-end flow:
1. Clone the repo
2. Install `ml_service` + `ml_training` in editable mode
3. Mount Drive, unzip SCIN
4. Inspect the CSV schema and auto-adapt
5. Precompute aligned face tensors
6. Train (resumable)
7. Export ONNX + download

**Before running** you need to have:
* pushed this repo to GitHub — replace `<YOUR_GITHUB_USER>` below
* uploaded `scin.zip` to `My Drive/skincare/scin/scin.zip`
* signed the SCIN DUA at https://github.com/google-research-datasets/scin

## 0. Confirm GPU

In [ ]:
!nvidia-smi | head -15

## 1. Clone repo + install
Replace `<YOUR_GITHUB_USER>` with your GitHub username. If the repo is private use a personal-access-token URL.

In [ ]:
GITHUB_USER = "moody23"
BRANCH = "main"

!git clone -b {BRANCH} https://github.com/moodykhalif23/facemeup.git /content/skincare
%cd /content/skincare/backend_v2
!pip install -q -e ml_service
!pip install -q -e ml_training
print("\nsetup done")

## 2. Mount Drive + unzip SCIN
Expected layout under `My Drive/skincare/scin/`:
```
scin.zip   → contains scin_cases.csv, scin_labels.csv, images/ (or scin_images/)
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/scin
!unzip -q -o /content/drive/MyDrive/skincare/scin/scin.zip -d /content/scin
!ls /content/scin | head

## 3. Inspect schema + auto-adapt column names
SCIN releases sometimes rename columns. The helper below renames whatever is closest to `case_id`, `condition`, and `confidence` so the parser in `ml_training.data.scin` picks them up without changes.

In [ ]:
import pandas as pd
from pathlib import Path

SCIN_ROOT = Path('/content/scin')
# Some bundles extract into a nested folder — walk one level down if needed.
if not (SCIN_ROOT / 'scin_cases.csv').is_file():
    candidates = [p for p in SCIN_ROOT.iterdir() if p.is_dir() and (p / 'scin_cases.csv').is_file()]
    if candidates:
        SCIN_ROOT = candidates[0]
print('SCIN root:', SCIN_ROOT)

cases_path = SCIN_ROOT / 'scin_cases.csv'
labels_path = SCIN_ROOT / 'scin_labels.csv'
cases = pd.read_csv(cases_path)
labels = pd.read_csv(labels_path)
print('cases cols:', cases.columns.tolist())
print('labels cols:', labels.columns.tolist())

def pick(cols, options):
    for c in cols:
        if c.lower() in options:
            return c
    return None

cases_id = pick(cases.columns, {'case_id', 'case', 'id'})
labels_id = pick(labels.columns, {'case_id', 'case', 'id'})
labels_cond = pick(labels.columns, {'condition', 'label', 'diagnosis'})
labels_conf = pick(labels.columns, {'confidence', 'weight', 'score'})
fit_col = pick(cases.columns, {'fitzpatrick', 'skin_type', 'fst'})

renames_cases = {}
renames_labels = {}
if cases_id and cases_id != 'case_id':   renames_cases[cases_id] = 'case_id'
if fit_col and fit_col != 'fitzpatrick': renames_cases[fit_col] = 'fitzpatrick'
if labels_id and labels_id != 'case_id': renames_labels[labels_id] = 'case_id'
if labels_cond and labels_cond != 'condition': renames_labels[labels_cond] = 'condition'
if labels_conf and labels_conf != 'confidence': renames_labels[labels_conf] = 'confidence'

if renames_cases:
    print('renaming in cases:', renames_cases)
    cases.rename(columns=renames_cases).to_csv(cases_path, index=False)
if renames_labels:
    print('renaming in labels:', renames_labels)
    labels.rename(columns=renames_labels).to_csv(labels_path, index=False)

# Also harmonise image folder name
img_dirs = [p for p in SCIN_ROOT.iterdir() if p.is_dir() and p.name.lower() in ('images', 'scin_images', 'img')]
if img_dirs and img_dirs[0].name != 'images':
    (SCIN_ROOT / 'images').symlink_to(img_dirs[0], target_is_directory=True) if not (SCIN_ROOT / 'images').exists() else None

print('schema normalised — SCIN root ready at', SCIN_ROOT)

## 4. Precompute aligned face tensors
Runs the `ml_service` preprocessing (face detect → align → CLAHE → save .npy) across every SCIN image. **~30–60 min** for ~10k images on Colab CPU cores. Fully resumable — re-run the cell to pick up where it left off.

In [ ]:
!python -m skin_training.data.precompute \
  --scin-root {SCIN_ROOT} \
  --output /content/work \
  --aligned-size 256 \
  --verbose 2>&1 | tail -20

In [ ]:
import json
print(json.dumps(json.loads(open('/content/work/index.json').read()), indent=2))
!echo 'labels.csv rows:'; wc -l /content/work/labels.csv
!echo 'aligned .npy count:'; ls /content/work/aligned | wc -l

## 5. Patch training config for Colab paths
If you want a faster/smaller first run, tune `epochs` or `batch_size` here.

In [ ]:
import yaml
from pathlib import Path

cfg = yaml.safe_load(Path('ml_training/configs/base.yaml').read_text())
cfg['data']['aligned_dir'] = '/content/work'
cfg['train']['checkpoint_dir'] = '/content/work/runs/exp1'
cfg['train']['batch_size'] = 32          # raise to 64 if T4 memory allows
cfg['train']['epochs']     = 25
cfg['train']['mixed_precision'] = True

Path('/content/config.yaml').write_text(yaml.safe_dump(cfg))
print(yaml.safe_dump(cfg))

## 6. Train
~40–80 min on a free T4 for 25 epochs over ~10k samples. Writes checkpoints to `/content/work/runs/exp1/`. Interrupt and re-run to resume from `last.pt`.

In [ ]:
!python -m skin_training.train.loop --config /content/config.yaml --verbose

## 7. TensorBoard (run in parallel with training)
You can execute this cell in a second Colab tab while training runs.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/work/runs/exp1/tb

## 8. Export to ONNX + numerical roundtrip check
Succeeds only when `|onnx - torch|_max < 1e-3`. If you hit a diff larger than that, tell me and I'll patch the export wrapper.

In [ ]:
!python -m skin_training.export.to_onnx \
  --checkpoint /content/work/runs/exp1/best.pt \
  --config /content/config.yaml \
  --output /content/work/skin_classifier_mobilenet.onnx \
  --verbose

## 9. Save back to Drive + download

In [ ]:
!mkdir -p /content/drive/MyDrive/skincare/artifacts
!cp /content/work/skin_classifier_mobilenet.onnx /content/drive/MyDrive/skincare/artifacts/
!cp /content/work/runs/exp1/best.pt /content/drive/MyDrive/skincare/artifacts/

from google.colab import files
files.download('/content/work/skin_classifier_mobilenet.onnx')

## 10. Local install — run on your machine, not Colab
```powershell
Copy-Item $HOME\Downloads\skin_classifier_mobilenet.onnx C:\Users\Sozuri\skincare\backend_v2\ml_service\models\
cd C:\Users\Sozuri\skincare\backend_v2
docker compose build ml-service
docker compose up -d ml-service
curl http://localhost:8013/healthz   # expect models_loaded: true
```

Logs should show `classifier ONNX loaded: skin_classifier_mobilenet.onnx`. After that, hitting `POST /api/v1/analyze` with a real face photo returns `"inference_mode": "onnx_mobilenet"` (not `placeholder_phase1`).